In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

In [4]:
EmpleadosAttrition = pd.read_csv('empleadosRETO.csv')
EmpleadosAttrition.head()

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,...,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsInCurrentRole,YearsSinceLastPromotion,Attrition
0,50,Travel_Rarely,Research & Development,1 km,2,Medical,1,997,4,Male,...,22,4,3,80,32,1,2,4,1,No
1,36,Travel_Rarely,Research & Development,6 km,2,Medical,1,178,2,Male,...,20,4,4,80,7,0,3,2,0,No
2,21,Travel_Rarely,Sales,7 km,1,Marketing,1,1780,2,Male,...,13,3,2,80,1,3,3,0,1,Yes
3,52,Travel_Rarely,Research & Development,7 km,4,Life Sciences,1,1118,2,Male,...,19,3,4,80,18,4,3,6,4,No
4,33,Travel_Rarely,Research & Development,15 km,1,Medical,1,582,2,Male,...,12,3,4,80,15,2,4,6,7,Yes


In [5]:
# Eliminar columnas que no aportan al análisis
EmpleadosAttrition = EmpleadosAttrition.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'])

In [6]:
# Calcular los años que el empleado lleva en la compañía hasta 2018
EmpleadosAttrition['Year'] = EmpleadosAttrition['HiringDate'].astype(str).str[-4:].astype(int)
EmpleadosAttrition['YearsAtCompany'] = 2018 - EmpleadosAttrition['Year']

In [7]:
# Convertir DistanceFromHome a número entero
EmpleadosAttrition = EmpleadosAttrition.rename(columns={'DistanceFromHome':'DistanceFromHome_km'})
EmpleadosAttrition['DistanceFromHome'] = EmpleadosAttrition['DistanceFromHome_km'].str.replace(' km','', regex=False).astype(int)

In [8]:
EmpleadosAttrition = EmpleadosAttrition.drop(columns=['Year', 'HiringDate', 'DistanceFromHome_km'])

In [9]:
# Tabla informativa de sueldo promedio por departamento
SueldoPromedioDepto = EmpleadosAttrition.groupby('Department')['MonthlyIncome'].mean().reset_index()
SueldoPromedioDepto = SueldoPromedioDepto.rename(columns={'MonthlyIncome':'SueldoPromedio'})
SueldoPromedioDepto

,Department,SueldoPromedio
0,Human Resources,6239.888889
1,Research & Development,6804.149813
2,Sales,7188.250000


In [10]:
# Escalar MonthlyIncome entre 0 y 1
escalador = MinMaxScaler()
EmpleadosAttrition['MonthlyIncome'] = escalador.fit_transform(EmpleadosAttrition[['MonthlyIncome']])

In [11]:
# Convertir variables categóricas a numéricas
categoricas = EmpleadosAttrition.select_dtypes(include='object').columns
codificador = LabelEncoder()

for columna in categoricas:
    EmpleadosAttrition[columna] = codificador.fit_transform(EmpleadosAttrition[columna])

In [12]:
# Correlación lineal de cada variable con Attrition
CorrelacionAttrition = EmpleadosAttrition.corr()['Attrition'].sort_values(ascending=False)
CorrelacionAttrition

,Attrition
Attrition,1.000000
OverTime,0.324777
MaritalStatus,0.187283
JobRole,0.078684
BusinessTravel,0.060677
Department,0.054236
DistanceFromHome,0.052732
EducationField,0.051184
PerformanceRating,-0.006471
NumCompaniesWorked,-0.009082


In [13]:
# Seleccionar variables con correlación mayor o igual a 0.1
variables_seleccionadas = CorrelacionAttrition[abs(CorrelacionAttrition) >= 0.1].index.tolist()

if 'Attrition' not in variables_seleccionadas:
    variables_seleccionadas.append('Attrition')

EmpleadosAttritionFinal = EmpleadosAttrition[variables_seleccionadas].copy()
EmpleadosAttritionFinal.head()

,Attrition,OverTime,MaritalStatus,EnvironmentSatisfaction,JobSatisfaction,JobInvolvement,YearsAtCompany,MonthlyIncome,YearsInCurrentRole,Age,TotalWorkingYears,JobLevel
0,0,0,0,4,4,3,5,0.864269,4,50,32,4
1,0,0,0,2,2,3,3,0.207340,2,36,7,2
2,1,0,2,2,2,3,1,0.088062,0,21,1,1
3,0,0,2,2,2,3,8,0.497574,6,52,18,3
4,1,1,1,2,3,3,7,0.664470,6,33,15,3


In [14]:
# Crear componentes principales y agregar los necesarios para explicar al menos el 80% de la varianza
pca = PCA()
EmpleadosAttritionPCA = pca.fit_transform(EmpleadosAttritionFinal)

varianza_acumulada = pca.explained_variance_ratio_.cumsum()
num_componentes = (varianza_acumulada < 0.80).sum() + 1

for i in range(num_componentes):
    EmpleadosAttritionFinal = EmpleadosAttritionFinal.assign(**{f'C{i}': EmpleadosAttritionPCA[:, i]})

EmpleadosAttritionFinal.head()

,Attrition,OverTime,MaritalStatus,EnvironmentSatisfaction,JobSatisfaction,JobInvolvement,YearsAtCompany,MonthlyIncome,YearsInCurrentRole,Age,TotalWorkingYears,JobLevel,C0,C1
0,0,0,0,4,4,3,5,0.864269,4,50,32,4,20.248685,-4.208533
1,0,0,0,2,2,3,3,0.207340,2,36,7,2,-6.380063,-3.688547
2,1,0,2,2,2,3,1,0.088062,0,21,1,1,-21.511025,1.740857
3,0,0,2,2,2,3,8,0.497574,6,52,18,3,13.824170,-5.843330
4,1,1,1,2,3,3,7,0.664470,6,33,15,3,-1.436496,4.144487


In [15]:
EmpleadosAttritionFinal.to_csv('EmpleadosAttritionFinal.csv', index=False)